In [57]:
import pandas as pd
import json
import torch
import torch.nn as nn
import transformers
from torch.utils.data import TensorDataset
from transformers.data.processors.utils import InputExample
from transformers.data.processors.glue import glue_convert_examples_to_features

# Load FairLex data

In [58]:
from datasets import load_dataset 

dataset = load_dataset("coastalcph/fairlex", "ecthr")

08/29/2025 14:49:35:WARNING:Reusing dataset fairlex (/home/lcorbucci/.cache/huggingface/datasets/coastalcph___fairlex/ecthr/1.0.0/b755f714459ab788a8e3f9167fe7463f79981775296915d36ac10fc58ea93737)
100%|██████████| 3/3 [00:00<00:00, 680.27it/s]


In [59]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'applicant_age', 'applicant_gender', 'defendant_state'],
        num_rows: 9000
    })
    test: Dataset({
        features: ['text', 'labels', 'applicant_age', 'applicant_gender', 'defendant_state'],
        num_rows: 1000
    })
    validation: Dataset({
        features: ['text', 'labels', 'applicant_age', 'applicant_gender', 'defendant_state'],
        num_rows: 1000
    })
})

In [60]:
# convert dataset to pandas dataframe 
train_dataset = dataset["train"].to_pandas()
val_dataset = dataset["validation"].to_pandas()
test_dataset = dataset["test"].to_pandas()

train_dataset.head(100)

,text,labels,applicant_age,applicant_gender,defendant_state
0,11. At the beginning of the events relevant t...,[4],0,0,1
1,9. The applicant is the monarch of Liechtenst...,[],2,1,1
2,9. In June 1949 plots of agricultural land ow...,[3],0,1,0
3,"8. In 1991 Mr Dušan Slobodník, a research wor...",[6],0,1,0
4,"9. The applicant is an Italian citizen, born ...",[],2,1,1
...,...,...,...,...,...
95,8. The applicants teach as temporary staff at...,"[3, 9]",0,0,1
96,8. The applicant was born in 1904 and lived i...,[3],3,1,0
97,9. Under the General Tax Code as worded until...,[9],0,0,1
98,8. Following complaints from the National Uni...,[4],0,0,1


In [63]:
print(train_dataset["applicant_gender"].value_counts())
print(train_dataset["defendant_state"].value_counts())

applicant_gender
1    4407
0    3306
2    1287
Name: count, dtype: int64
defendant_state
0    7224
1    1776
Name: count, dtype: int64


# Load Data

In [2]:
def parse_input(file_name):
    json_files = []
    with open('train.json') as f:
        for line in f: 
            json_files.append(json.loads(line))

    data = []
    for json_file in json_files:
        sentence = " ".join(json_file['sentence'])
        target = json_file['gender']
        data.append((sentence, target))
    
    return pd.DataFrame(data, columns=['text', 'target'])

In [3]:
train_df = parse_input('train.json')
test_df = parse_input("test.json")

In [4]:
train_df.head(100)

,text,target
0,"An old man answers the advertisement , claimin...",male
1,"An old woman answers the advertisement , claim...",female
2,"As he leaves , he realises that he has a lame ...",male
3,"As she leaves , she realises that she has a la...",female
4,He says he has sounded its depths and located ...,male
...,...,...
95,She emphasises that she only used her African ...,female
96,On the day in 1815 when he escapes the Island ...,male
97,On the day in 1815 when she escapes the Island...,female
98,"Reluctantly , he orders the harpooneers to ins...",male


In [5]:
train_df["text"] = train_df["text"].astype(str)
train_df["target"] = train_df["target"].astype(str)

In [6]:
test_df.head(100)

,text,target
0,"An old man answers the advertisement , claimin...",male
1,"An old woman answers the advertisement , claim...",female
2,"As he leaves , he realises that he has a lame ...",male
3,"As she leaves , she realises that she has a la...",female
4,He says he has sounded its depths and located ...,male
...,...,...
95,She emphasises that she only used her African ...,female
96,On the day in 1815 when he escapes the Island ...,male
97,On the day in 1815 when she escapes the Island...,female
98,"Reluctantly , he orders the harpooneers to ins...",male


In [7]:
test_df["target"].value_counts()

target
male      1288
female    1288
Name: count, dtype: int64

# Model

In [8]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_id = "answerdotai/ModernBERT-base"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2)
model.classifier = torch.nn.Linear(in_features=768, out_features=2)


Some weights of ModernBertForSequenceClassification were not initialized from the model checkpoint at answerdotai/ModernBERT-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [9]:
trainable_layers = [model.model.layers[-1], model.head, model.classifier]

total_params = 0
trainable_params = 0

for p in model.parameters():
        p.requires_grad = False
        total_params += p.numel()

for layer in trainable_layers:
    for p in layer.parameters():
        p.requires_grad = True
        trainable_params += p.numel()

print(f"Total parameters count: {total_params:,}") # ~150
print(f"Trainable parameters count: {trainable_params:,}") # ~5.6

Total parameters count: 149,606,402
Trainable parameters count: 5,607,170


In [10]:
from peft import get_peft_model, LoraConfig, TaskType

LORA_TARGET_MODULES = ["Wqkv"]  # ModernBERT combined QKV layer

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,  # our particular task is sequence classification
    inference_mode=False,  # Enable training mode
    r=32,  # Low-rank dimension
    lora_alpha=32,  # Alpha scaling factor
    lora_dropout=0.05,  # Dropout for LoRA layers
    target_modules=LORA_TARGET_MODULES,
)

model_with_lora = get_peft_model(model, lora_config)
trainable_params = sum(p.numel() for p in model_with_lora.parameters() if p.requires_grad)
print(f"Total trainable parameters with LoRA: {trainable_params:,}") # ~1M

Total trainable parameters with LoRA: 2,164,226


In [11]:

# Count trainable parameters
trainable_params = sum(p.numel() for p in model_with_lora.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model_with_lora.parameters())

print(f"\nParameter Summary:")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Frozen parameters: {total_params - trainable_params:,}")
print(f"Trainable percentage: {100 * trainable_params / total_params:.2f}%")

# Optional: Print detailed breakdown of trainable parameters
print(f"\nDetailed breakdown:")
lora_params = sum(p.numel() for name, p in model_with_lora.named_parameters() 
                  if p.requires_grad and 'lora' in name.lower())
classifier_params = sum(p.numel() for name, p in model_with_lora.named_parameters() 
                       if p.requires_grad and 'classifier' in name.lower())
last_layer_params = sum(p.numel() for name, p in model_with_lora.named_parameters() 
                       if p.requires_grad and 'encoder.layer.21' in name)

print(f"LoRA parameters: {lora_params:,}")
print(f"Classifier parameters: {classifier_params:,}")
print(f"Last layer (non-LoRA) parameters: {last_layer_params:,}")
print(f"Other trainable parameters: {trainable_params - lora_params - classifier_params - last_layer_params:,}")


Parameter Summary:
Total parameters: 151,770,628
Trainable parameters: 2,164,226
Frozen parameters: 149,606,402
Trainable percentage: 1.43%

Detailed breakdown:
LoRA parameters: 2,162,688
Classifier parameters: 1,538
Last layer (non-LoRA) parameters: 0
Other trainable parameters: 0


In [12]:
model = model_with_lora

# Prepare the data

In [13]:
import pandas as pd
import torch
import transformers
from torch.utils.data import TensorDataset
from transformers.data.processors.utils import InputExample
from transformers.data.processors.glue import glue_convert_examples_to_features

LABEL_LIST = ['male', 'female']
MAX_SEQ_LENGTH = 128

def create_examples(df, set_type):
    """ Convert raw dataframe to a list of InputExample. Filter malformed examples
    """
    examples = []
    skipped = 0
    
    for index, row in df.iterrows():
        # Validate target
        target_str = str(row['target']).strip()
        if target_str not in LABEL_LIST:
            skipped += 1
            continue
        
        # Validate and clean text
        text = row['text']
        if text is None or pd.isna(text):
            skipped += 1
            continue
            
        # Convert to string and clean
        text = str(text).strip()
        if not text:  # Skip empty strings
            skipped += 1
            continue
        
        # Additional validation: ensure text doesn't contain problematic characters
        try:
            # Test if the text can be encoded/decoded properly
            text.encode('utf-8').decode('utf-8')
        except (UnicodeEncodeError, UnicodeDecodeError):
            skipped += 1
            continue
            
        guid = f"{index}-{set_type}"
        examples.append(
            InputExample(guid=guid, text_a=text, text_b=None, label=target_str)
        )
    
    print(f"Created {len(examples)} examples, skipped {skipped} invalid entries for {set_type}")
    return examples

def df_to_features_custom(df, set_type, tokenizer):
    """ Custom feature conversion that bypasses glue_convert_examples_to_features
    """
    examples = create_examples(df, set_type)
    
    if not examples:
        raise ValueError(f"No valid examples found in {set_type} dataset")
    
    print(f"Processing {len(examples)} examples for {set_type}")
    
    # Extract texts and labels
    texts = [example.text_a for example in examples]
    # Convert string labels to integers: 'male' -> 0, 'female' -> 1
    label_to_id = {'male': 0, 'female': 1}
    labels = [label_to_id[example.label] for example in examples]
    
    # Debug: print first few texts to check for issues
    print(f"Sample texts from {set_type}:")
    for i, text in enumerate(texts[:3]):
        print(f"  {i}: '{text}' (type: {type(text)}, length: {len(text)})")
    
    # Tokenize directly without using GLUE converter
    try:
        print(f"Tokenizing {len(texts)} texts...")
        encoded = tokenizer(
            texts,
            add_special_tokens=True,
            max_length=MAX_SEQ_LENGTH,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
            return_attention_mask=True,
            return_token_type_ids=True
        )
        print("Tokenization successful!")
        
        # Create feature objects for compatibility
        class SimpleFeature:
            def __init__(self, input_ids, attention_mask, token_type_ids, label):
                self.input_ids = input_ids
                self.attention_mask = attention_mask
                self.token_type_ids = token_type_ids
                self.label = label
        
        features = []
        for i in range(len(texts)):
            features.append(SimpleFeature(
                input_ids=encoded['input_ids'][i].tolist(),
                attention_mask=encoded['attention_mask'][i].tolist(),
                token_type_ids=encoded['token_type_ids'][i].tolist() if 'token_type_ids' in encoded else [0] * MAX_SEQ_LENGTH,
                label=labels[i]
            ))
        
        return features
        
    except Exception as e:
        print(f"Error during tokenization: {e}")
        print("Debugging problematic texts...")
        
        # Find problematic texts
        for i, text in enumerate(texts):
            try:
                tokenizer(text, max_length=10, truncation=True)
            except Exception as text_error:
                print(f"Problematic text at index {i}: '{text}' -> {text_error}")
                if i >= 5:  # Don't print too many
                    break
        raise

def features_to_dataset(features):
    """ Convert features into a single dataset
    """
    all_input_ids = torch.tensor([f.input_ids for f in features], dtype=torch.long)
    all_attention_mask = torch.tensor([f.attention_mask for f in features], dtype=torch.long)
    all_token_type_ids = torch.tensor([f.token_type_ids for f in features], dtype=torch.long)
    all_labels = torch.tensor([f.label for f in features], dtype=torch.long)
    
    dataset = TensorDataset(
        all_input_ids, all_attention_mask, all_token_type_ids, all_labels
    )
    
    return dataset

# Data quality checks
print("Checking data quality...")
print(f"Train dataset shape: {train_df.shape}")
print(f"Test dataset shape: {test_df.shape}")

# Check for missing values
print(f"Train text missing values: {train_df['text'].isna().sum()}")
print(f"Train target missing values: {train_df['target'].isna().sum()}")

# Check unique targets
print(f"Unique targets in train: {train_df['target'].unique()}")

# Check for any weird text entries
print("Sample text entries:")
for i in range(min(3, len(train_df))):
    text = train_df.iloc[i]['text']
    print(f"  {i}: '{text}' (type: {type(text)})")

# Process the datasets using custom function
print("\n" + "="*50)
print("Processing datasets...")
train_features = df_to_features_custom(train_df, "train", tokenizer)
test_features = df_to_features_custom(test_df, "test", tokenizer)

train_dataset = features_to_dataset(train_features)
test_dataset = features_to_dataset(test_features)

print(f"Successfully created datasets!")
print(f"Train dataset size: {len(train_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")

Checking data quality...
Train dataset shape: (2576, 2)
Test dataset shape: (2576, 2)
Train text missing values: 0
Train target missing values: 0
Unique targets in train: ['male' 'female']
Sample text entries:
  0: 'An old man answers the advertisement , claiming that the ring belongs to his son .' (type: <class 'str'>)
  1: 'An old woman answers the advertisement , claiming that the ring belongs to her daughter .' (type: <class 'str'>)
  2: 'As he leaves , he realises that he has a lame leg , and believes this is the reason he has been ‘ left on the shelf ’ .' (type: <class 'str'>)

Processing datasets...
Created 2576 examples, skipped 0 invalid entries for train
Processing 2576 examples for train
Sample texts from train:
  0: 'An old man answers the advertisement , claiming that the ring belongs to his son .' (type: <class 'str'>, length: 82)
  1: 'An old woman answers the advertisement , claiming that the ring belongs to her daughter .' (type: <class 'str'>, length: 89)
  2: 'As he 

# Training

In [14]:
BATCH_SIZE = 64
MAX_PHYSICAL_BATCH_SIZE = 32

from torch.utils.data import DataLoader, RandomSampler, SequentialSampler
from opacus.utils.uniform_sampler import UniformWithReplacementSampler

train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE)
test_dataloader = DataLoader(test_dataset, sampler=SequentialSampler(test_dataset), batch_size=BATCH_SIZE)

In [15]:
# Move the model to appropriate device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Set the model to train mode (HuggingFace models load in eval mode)
model = model.train()
# Define optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, eps=1e-8)


In [16]:
EPOCHS = 5
LOGGING_INTERVAL = 5 # once every how many steps we run evaluation cycle and report metrics
EPSILON = 7.5
DELTA = 1 / len(train_dataloader) # Parameter for privacy accounting. Probability of not achieving privacy guarantees

In [17]:
import numpy as np
from tqdm.notebook import tqdm

def accuracy(preds, labels):
    return (preds == labels).mean()

# define evaluation cycle
def evaluate(model):
    model.eval()

    loss_arr = []
    accuracy_arr = []

    all_predictions = []
    all_labels = []

    for batch in test_dataloader:
        batch = tuple(t.to(device) for t in batch)

        with torch.no_grad():
            inputs = {'input_ids':      batch[0],
                      'attention_mask': batch[1],
                      'token_type_ids': batch[2],
                      'labels':         batch[3]}

            outputs = model(**inputs)
            loss, logits = outputs[:2]

            preds = np.argmax(logits.detach().cpu().numpy(), axis=1)
            labels = inputs['labels'].detach().cpu().numpy()

            loss_arr.append(loss.item())
            accuracy_arr.append(accuracy(preds, labels))

            all_predictions.extend(list(preds))
            all_labels.extend(list(labels))

    all_predictions = torch.tensor([int(x) for x in all_predictions])
    all_labels = torch.tensor([int(x) for x in all_labels])

    model.train()
    return np.mean(loss_arr), np.mean(accuracy_arr), all_predictions, all_labels

In [18]:
from opacus import PrivacyEngine

MAX_GRAD_NORM = 0.1

privacy_engine = PrivacyEngine()

model, optimizer, criterion, train_dataloader = privacy_engine.make_private_with_epsilon(
    module=model,
    optimizer=optimizer,
    data_loader=train_dataloader,
    target_delta=DELTA,
    target_epsilon=EPSILON,
    epochs=EPOCHS,
    max_grad_norm=MAX_GRAD_NORM,
    grad_sample_mode="ghost",
)

/home/lcorbucci/PUFFLE/.venv/lib/python3.12/site-packages/opacus/privacy_engine.py:96: UserWarning: Secure RNG turned off. This is perfectly fine for experimentation as it allows for much faster training performance, but remember to turn it on and retrain one last time before production with ``secure_mode`` turned on.
  warnings.warn(
/home/lcorbucci/PUFFLE/.venv/lib/python3.12/site-packages/opacus/accountants/analysis/rdp.py:332: UserWarning: Optimal order is the largest alpha. Please consider expanding the range of alphas to get a tighter privacy bound.
  warnings.warn(


In [19]:
from opacus.utils.batch_memory_manager import BatchMemoryManager
from puffle.Regularization.disparity_loss import DisparityRegularizationLoss

for epoch in range(1, EPOCHS+1):
    losses = []

    with BatchMemoryManager(
        data_loader=train_dataloader,
        max_physical_batch_size=MAX_PHYSICAL_BATCH_SIZE,
        optimizer=optimizer
    ) as memory_safe_data_loader:
        for step, batch in enumerate(memory_safe_data_loader):
            optimizer.zero_grad()

            batch = tuple(t.to(device) for t in batch)
            inputs = {'input_ids':      batch[0],
                    'attention_mask': batch[1],
                    'token_type_ids': batch[2],
                    'labels':         batch[3]}
            labels = batch[3]
            outputs = model(**inputs) # output = loss, logits, hidden_states, attentions
            

            loss = outputs[0]
            loss.backward()
            losses.append(loss.item())
            
            optimizer.step()
            logits = outputs[1]
            argmax_logits = logits.argmax(dim=-1)

            

            if step > 0 and step % LOGGING_INTERVAL == 0:
                train_loss = np.mean(losses)
                eps = privacy_engine.get_epsilon(DELTA)

                eval_loss, eval_accuracy, preds, labels = evaluate(model)

                print(labels)
                print(preds)
                max_disparity = 0
                for current_target in [0, 1]:
                    for current_sensitive_attribute in [0, 1]:
                        current_disparity = DisparityRegularizationLoss().compute_violation_with_argmax(
                            predictions_argmax = preds,
                            sensitive_attribute_list = labels,
                            current_target = current_target, 
                            current_sensitive_feature = current_sensitive_attribute
                        )
                        max_disparity = max(current_disparity, max_disparity)

                print(
                  f"Epoch: {epoch} | "
                  f"Step: {step} | "
                  f"Train loss: {train_loss:.3f} | "
                  f"Eval loss: {eval_loss:.3f} | "
                  f"Eval accuracy: {eval_accuracy:.3f} | "
                  f"ɛ: {eps:.2f} | "
                  f"Max Disparity: {max_disparity:.3f}"
                )

tensor([0, 1, 0,  ..., 1, 0, 1])
tensor([0, 1, 0,  ..., 1, 1, 1])
Y_eq_k_and_Z_eq_z 0 0: 889 - Z_eq_z: 1288 - Y_eq_k_and_Z_not_eq_z: 556 - Z_not_eq_z: 1288
Y_eq_k_and_Z_eq_z 0 1: 556 - Z_eq_z: 1288 - Y_eq_k_and_Z_not_eq_z: 889 - Z_not_eq_z: 1288
Y_eq_k_and_Z_eq_z 1 0: 399 - Z_eq_z: 1288 - Y_eq_k_and_Z_not_eq_z: 732 - Z_not_eq_z: 1288
Y_eq_k_and_Z_eq_z 1 1: 732 - Z_eq_z: 1288 - Y_eq_k_and_Z_not_eq_z: 399 - Z_not_eq_z: 1288
Epoch: 1 | Step: 5 | Train loss: 0.683 | Eval loss: 0.658 | Eval accuracy: 0.628 | ɛ: 0.08 | Max Disparity: 0.259
tensor([0, 1, 0,  ..., 1, 0, 1])
tensor([1, 1, 0,  ..., 1, 1, 1])
Y_eq_k_and_Z_eq_z 0 0: 605 - Z_eq_z: 1288 - Y_eq_k_and_Z_not_eq_z: 145 - Z_not_eq_z: 1288
Y_eq_k_and_Z_eq_z 0 1: 145 - Z_eq_z: 1288 - Y_eq_k_and_Z_not_eq_z: 605 - Z_not_eq_z: 1288
Y_eq_k_and_Z_eq_z 1 0: 683 - Z_eq_z: 1288 - Y_eq_k_and_Z_not_eq_z: 1143 - Z_not_eq_z: 1288
Y_eq_k_and_Z_eq_z 1 1: 1143 - Z_eq_z: 1288 - Y_eq_k_and_Z_not_eq_z: 683 - Z_not_eq_z: 1288
Epoch: 1 | Step: 10 | Train loss